# TP 2 · Couche Silver avec Spark et Apache Iceberg

## Étape 2 · Créer la table Silver et écrire les données nettoyées

In [1]:
import time
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("tp2").getOrCreate()

# Utilitaire : relève fichiers / snapshots / lignes après chaque étape
results = []
def stats(label):
    nb_f = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.files").collect()[0].c
    nb_s = spark.sql("SELECT count(*) AS c FROM demo.silver.orders.snapshots").collect()[0].c
    nb_l = spark.table("demo.silver.orders").count()
    results.append((label, nb_f, nb_s, nb_l))
    print(f"[{label}] fichiers: {nb_f} | snapshots: {nb_s} | lignes: {nb_l}")

26/10/17 16:09:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Cellule 1 · Espace de noms et table

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.orders (
  order_id STRING, customer_id BIGINT, order_ts TIMESTAMP,
  city STRING, amount DECIMAL(10,2), status STRING,
  _ingested_at TIMESTAMP)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES ('write.parquet.compression-codec' = 'zstd')
""")

DataFrame[]

### Cellule 2 · Nettoyer et écrire

In [11]:
raw = spark.read.parquet("/home/iceberg/notebooks/data/orders/")
silver = (raw
    .dropDuplicates(["order_id"])                       # unicité
    .filter(F.col("amount") > 0)                        # validité
    .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
    .withColumn("city", F.initcap(F.trim("city")))      # normalisation
    .withColumn("status", F.upper("status"))
    .drop("_batch_id"))
silver.writeTo("demo.silver.orders").append()
spark.table("demo.silver.orders").groupBy("status").count().show()

[Stage 103:==================================================>  (108 + 6) / 114]

+---------+------+
|   status| count|
+---------+------+
|     PAID|333860|
|CANCELLED|332865|
|  SHIPPED|333275|
+---------+------+



### Cellule 3 · Tables de métadonnées

In [13]:
spark.sql("SELECT * FROM demo.silver.orders.snapshots").show(truncate=False)
spark.sql("""SELECT partition, record_count, file_size_in_bytes
             FROM demo.silver.orders.files LIMIT 10""").show(truncate=False)
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
stats("Étape 2 : après écriture Silver")

+-----------------------+-------------------+-------------------+---------+----------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                             |summary                                                                                                                                                    

### Questions 1 à 3

In [14]:
# Q1 : nombre de partitions (jours) distinctes
spark.sql("SELECT count(*) AS nb_partitions FROM demo.silver.orders.partitions").show()

+-------------+
|nb_partitions|
+-------------+
|           91|
+-------------+



In [15]:
# Q2 : contenu du metadata.json courant (via le catalogue REST)
import requests, json
try:
    r = requests.get("http://rest:8181/v1/namespaces/silver/tables/orders").json()
    m = r["metadata"]
    print("fichier metadata :", r["metadata-location"])
    print("snapshot courant :", m["current-snapshot-id"])
    print("schéma           :", [(f["id"], f["name"], f["type"]) for f in m["schemas"][-1]["fields"]])
    print("partitionnement  :", json.dumps(m["partition-specs"][-1]))
except Exception as e:
    print("Lecture via REST impossible :", e)
    print("-> télécharger le dernier *.metadata.json depuis la console MinIO (warehouse/silver/orders/metadata/)")

fichier metadata : s3://warehouse/silver/orders/metadata/00005-f433b091-c48d-42a0-b514-863c105e36b9.metadata.json
snapshot courant : 1828977901580180334
schéma           : [(1, 'order_id', 'string'), (2, 'customer_id', 'long'), (3, 'order_ts', 'timestamptz'), (4, 'city', 'string'), (5, 'amount', 'decimal(10, 2)'), (6, 'status', 'string'), (7, '_ingested_at', 'timestamptz')]
partitionnement  : {"spec-id": 0, "fields": [{"name": "order_ts_day", "transform": "day", "source-id": 3, "field-id": 1000}]}


In [16]:
# Q3 : plan d'exécution avec filtre sur order_ts (colonne source)
plan = spark.sql("""EXPLAIN SELECT count(*) FROM demo.silver.orders
                    WHERE order_ts >= current_date() - INTERVAL 7 DAYS""").collect()[0][0]
print(plan)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(agg_func_0#2178L)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1917]
      +- HashAggregate(keys=[], functions=[partial_sum(agg_func_0#2178L)])
         +- Project [count(*)#2179L AS agg_func_0#2178L]
            +- LocalTableScan [count(*)#2179L]




## Étape 3 · Upsert avec MERGE INTO

In [19]:
spark.sql("SELECT count(*) AS nb_fichiers FROM demo.silver.orders.files").show()
spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").show(truncate=False)

def make_updates():
    """1 000 commandes PAID -> SHIPPED + 500 nouvelles commandes (figé avec localCheckpoint)"""
    df = spark.sql("""
        SELECT * FROM (
          SELECT order_id, customer_id, order_ts, city, amount,
                 'SHIPPED' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders WHERE status = 'PAID' LIMIT 1000)
        UNION ALL
        SELECT * FROM (
          SELECT uuid() AS order_id, customer_id, current_timestamp() AS order_ts, city, amount,
                 'PAID' AS status, current_timestamp() AS _ingested_at
          FROM demo.silver.orders LIMIT 500)
    """).localCheckpoint()
    df.createOrReplaceTempView("updates")
    return df

MERGE_SQL = """
MERGE INTO demo.silver.orders t
USING updates s
ON t.order_id = s.order_id
WHEN MATCHED AND t.status <> s.status THEN
  UPDATE SET t.status = s.status, t._ingested_at = s._ingested_at
WHEN NOT MATCHED THEN INSERT *
"""

+-----------+
|nb_fichiers|
+-----------+
|        455|
+-----------+

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|7092805249271166307|2026-10-17 16:10:55.295|append   |
|630320151308632667 |2026-10-17 16:11:19.79 |append   |
|1049987017696210065|2026-10-17 16:11:55.973|append   |
|6430183226957363104|2026-10-17 16:12:11.647|append   |
|1828977901580180334|2026-10-17 16:12:28.553|append   |
+-------------------+-----------------------+---------+



### MERGE en copy-on-write (mode par défaut)

In [28]:
spark.sql("""SELECT count(*) AS lignes, count(DISTINCT order_id) AS ids_distincts
             FROM demo.silver.orders""").show()

[Stage 213:======================================>                  (2 + 1) / 3]

+-------+-------------+
| lignes|ids_distincts|
+-------+-------------+
|1000001|       200001|
+-------+-------------+



In [20]:
upd1 = make_updates()
t0 = time.time()
spark.sql(MERGE_SQL)
t_cow = time.time() - t0
print(f"MERGE copy-on-write : {t_cow:.1f} s")
stats("Étape 3 : après MERGE copy-on-write")
spark.sql("""SELECT snapshot_id, committed_at, operation,
                    summary['added-records'], summary['deleted-records'],
                    summary['added-data-files'], summary['deleted-data-files']
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)

26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/05/temp_shuffle_f356a499-6867-4ea9-9847-b02ed5236c49
26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/3b/temp_shuffle_9e635c25-3660-4f80-bc45-bbcb073a7376
26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/27/temp_shuffle_825f2d6e-150c-4458-a867-2935057a95ac
26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/09/temp_shuffle_bf7ca98a-f417-49a4-a3a0-e80fc9d6b0a4
26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/21/temp_shuffle_d88c8124-8983-4a91-836a-4ac58c6e9894
26/10/17 16:23:05 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-09fb749d-50eb-4af8-9816-63e874d03fea/09/temp_shuffle_43f4de8f-d96b-4eaf-a99a-ef9c72105e88
26/10/17 1

Py4JJavaError: An error occurred while calling o36.sql.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 170.0 failed 1 times, most recent failure: Lost task 0.0 in stage 170.0 (TID 1106) (91ad28e4ebbe executor driver): org.apache.spark.SparkRuntimeException: [MERGE_CARDINALITY_VIOLATION] The ON search condition of the MERGE statement matched a single row from the target table with multiple rows of the source table.
This could result in the target row being operated on more than once with an update or delete operation and is not allowed.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.mergeCardinalityViolationError(QueryExecutionErrors.scala:2702)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$BitmapCardinalityValidator.validate(MergeRowsExec.scala:164)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$MergeRowIterator.next(MergeRowsExec.scala:203)
	at org.apache.spark.sql.execution.datasources.v2.MergeRowsExec$Me

### Expérience : merge-on-read

In [ ]:
spark.sql("""ALTER TABLE demo.silver.orders SET TBLPROPERTIES (
  'write.merge.mode' = 'merge-on-read',
  'write.update.mode' = 'merge-on-read',
  'write.delete.mode' = 'merge-on-read')""")
upd2 = make_updates()          # nouveau lot : 1 000 autres PAID + 500 nouvelles commandes
t0 = time.time()
spark.sql(MERGE_SQL)
t_mor = time.time() - t0
print(f"MERGE merge-on-read : {t_mor:.1f} s")
stats("Étape 3 : après MERGE merge-on-read")
spark.sql("""SELECT snapshot_id, operation,
                    summary['added-data-files'] AS added_data, summary['deleted-data-files'] AS deleted_data,
                    summary['added-delete-files'] AS added_delete_files,
                    summary['added-position-deletes'] AS pos_deletes
             FROM demo.silver.orders.snapshots ORDER BY committed_at""").show(truncate=False)
spark.sql("SELECT content, count(*) AS nb FROM demo.silver.orders.files GROUP BY content").show()

### Idempotence : rejouer le MERGE avec la même vue

In [ ]:
before = spark.table("demo.silver.orders").count()
spark.sql(MERGE_SQL)          # même vue 'updates' (upd2, en cache)
after = spark.table("demo.silver.orders").count()
print("lignes avant :", before, "| après :", after)
stats("Étape 3 : après re-MERGE identique")
spark.sql("""SELECT snapshot_id, operation, summary['added-records'], summary['changed-partition-count']
             FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1""").show(truncate=False)

## Étape 4 · Time travel, rollback et évolution du schéma

In [ ]:
snaps = spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.silver.orders.snapshots ORDER BY committed_at").collect()
for s in snaps: print(s.snapshot_id, s.committed_at, s.operation)
snap_before_merge = snaps[0].snapshot_id          # l'append initial, avant tout MERGE
ts_before_merge = snaps[0].committed_at
print("snapshot avant MERGE :", snap_before_merge)

In [ ]:
print("--- état AVANT le MERGE (VERSION AS OF) ---")
spark.sql(f"SELECT status, count(*) AS nb FROM demo.silver.orders VERSION AS OF {snap_before_merge} GROUP BY status ORDER BY status").show()
print("--- état ACTUEL ---")
spark.sql("SELECT status, count(*) AS nb FROM demo.silver.orders GROUP BY status ORDER BY status").show()
spark.sql(f"SELECT count(*) AS nb FROM demo.silver.orders TIMESTAMP AS OF '{ts_before_merge}'").show()

### Erreur simulée puis rollback

In [ ]:
snap_before_delete = spark.sql("SELECT snapshot_id FROM demo.silver.orders.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0].snapshot_id
n_before = spark.table("demo.silver.orders").count()
spark.sql("DELETE FROM demo.silver.orders WHERE status = 'SHIPPED'")
n_deleted = spark.table("demo.silver.orders").count()
print("lignes avant DELETE :", n_before, "| après DELETE :", n_deleted)
spark.sql(f"CALL demo.system.rollback_to_snapshot('silver.orders', {snap_before_delete})").show()
print("lignes après rollback :", spark.table("demo.silver.orders").count())
stats("Étape 4 : après DELETE + rollback")

### Évolution du schéma (sans réécriture)

In [21]:
spark.sql("ALTER TABLE demo.silver.orders ADD COLUMN channel STRING")
spark.sql("ALTER TABLE demo.silver.orders RENAME COLUMN city TO delivery_city")
spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(12,2)")
spark.sql("DESCRIBE TABLE demo.silver.orders").show(truncate=False)
spark.sql("SELECT order_id, delivery_city, amount, channel FROM demo.silver.orders LIMIT 5").show()
stats("Étape 4 : après évolution du schéma")

+--------------+--------------+-------+
|col_name      |data_type     |comment|
+--------------+--------------+-------+
|order_id      |string        |NULL   |
|customer_id   |bigint        |NULL   |
|order_ts      |timestamp     |NULL   |
|delivery_city |string        |NULL   |
|amount        |decimal(12,2) |NULL   |
|status        |string        |NULL   |
|_ingested_at  |timestamp     |NULL   |
|channel       |string        |NULL   |
|              |              |       |
|# Partitioning|              |       |
|Part 0        |days(order_ts)|       |
+--------------+--------------+-------+



+--------------------+--------------------+------+-------+
|            order_id|       delivery_city|amount|channel|
+--------------------+--------------------+------+-------+
|007f8431-ebcd-407...|   Vaillant-la-forêt|407.76|   NULL|
|0095483d-ac12-420...|Saint Maurice-les...|736.32|   NULL|
|0221b4a7-e618-44b...|              Potier|136.12|   NULL|
|035dd2c5-2ec8-428...|            Denisnec|854.19|   NULL|
|04ad3298-da3f-414...|             Renault|194.35|   NULL|
+--------------------+--------------------+------+-------+

[Étape 4 : après évolution du schéma] fichiers: 455 | snapshots: 5 | lignes: 1000000


In [22]:
# Test : réduire DECIMAL(12,2) -> DECIMAL(10,2) doit être refusé
try:
    spark.sql("ALTER TABLE demo.silver.orders ALTER COLUMN amount TYPE DECIMAL(10,2)")
except Exception as e:
    print("Refusé comme prévu :", str(e)[:300])

Refusé comme prévu : [NOT_SUPPORTED_CHANGE_COLUMN] ALTER TABLE ALTER/CHANGE COLUMN is not supported for changing `demo`.`silver`.`orders`'s column `amount` with type "DECIMAL(12,2)" to `amount` with type "DECIMAL(10,2)".; line 1 pos 0;
AlterColumn resolvedfieldname(StructField(amount,DecimalType(12,2),true)), DecimalTyp


### Évolution du partitionnement : jour -> mois

In [23]:
spark.sql("ALTER TABLE demo.silver.orders REPLACE PARTITION FIELD order_ts_day WITH months(order_ts)")
spark.sql("SELECT * FROM demo.silver.orders.partitions").show(truncate=False)

+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition         |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{2026-10-02, NULL}|0      |11645       |5         |391730                       |0                           |0                         |0                           |0                         |2026-10-17 16:12:28.553|1828977901580180334     |
|{2026-09-07, NULL}|0   

In [24]:
# Petit lot de test pour voir où vont les nouvelles données
spark.sql("""INSERT INTO demo.silver.orders
             SELECT uuid(), 1, current_timestamp(), 'Test-Ville', CAST(10.00 AS DECIMAL(12,2)), 'PAID', current_timestamp(), 'web'""")
spark.sql("SELECT partition, record_count FROM demo.silver.orders.files ORDER BY partition DESC LIMIT 5").show(truncate=False)
stats("Étape 4 : après changement de partitionnement")

+------------------+------------+
|partition         |record_count|
+------------------+------------+
|{2026-10-12, NULL}|1662        |
|{2026-10-12, NULL}|1662        |
|{2026-10-12, NULL}|1662        |
|{2026-10-12, NULL}|1662        |
|{2026-10-12, NULL}|1662        |
+------------------+------------+

[Étape 4 : après changement de partitionnement] fichiers: 456 | snapshots: 6 | lignes: 1000001


## Étape 5 · Maintenance et export

In [25]:
spark.sql("""CALL demo.system.rewrite_data_files(
  table => 'silver.orders',
  options => map('target-file-size-bytes', '134217728'))""").show(truncate=False)
stats("Étape 5 : après compaction")

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|455                       |4                     |33667440             |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

[Étape 5 : après compaction] fichiers: 5 | snapshots: 7 | lignes: 1000001


In [26]:
spark.sql("""CALL demo.system.expire_snapshots(
  table => 'silver.orders',
  older_than => current_timestamp(),
  retain_last => 5)""").show(truncate=False)
stats("Étape 5 : après expire_snapshots")

AnalysisException: 
mismatched input '(' expecting STRING
== SQL ==
CALL demo.system.expire_snapshots(
  table => 'silver.orders',
  older_than => current_timestamp(),
  retain_last => 5)

In [ ]:
spark.sql("CALL demo.system.remove_orphan_files(table => 'silver.orders')").show(truncate=False)
stats("Étape 5 : après remove_orphan_files")

In [ ]:
import os
os.makedirs("/home/iceberg/notebooks/data/silver", exist_ok=True)
df = spark.table("demo.silver.orders").toPandas()
df.to_parquet("/home/iceberg/notebooks/data/silver/orders.parquet", index=False)
print(len(df), "lignes exportées")

## Récapitulatif

In [ ]:
import pandas as pd
print(pd.DataFrame(results, columns=["étape", "fichiers", "snapshots", "lignes"]).to_string(index=False))
print(f"\nMERGE copy-on-write : {t_cow:.1f} s | merge-on-read : {t_mor:.1f} s")